# Genie 3 — S100B binder design (cluster)

De novo binder design against the **S100B homodimer** (Project Sentinel) with [aqlaboratory/genie3](https://github.com/aqlaboratory/genie3).

**How Genie 3 binder design works — two stages, not one hotspot YAML:**
1. **Prepare a problem set** with `scripts/problem/binder_design/prepare.py` — parses the target PDB, runs ColabFold to build the target MSA, and *computes* the interface by distance from a few seed hotspots. Writes a problem JSON.
2. **Run** `genie3 run` against that problem set, selecting the problem by key.

**Cluster assumptions for this notebook:**
- You've already done the one-time setup below (clone + `setup.sh` + `download.sh`).
- This **kernel is launched from the activated `genie3` conda env on a login node** — it needs internet (for the MSA step in `prepare.py`) and `sbatch`.
- The GPU-heavy generation is **submitted to the scheduler**, not run in the kernel.

> ⚠️ **No avoid/exclusion mask.** Genie 3 binder conditioning is purely positive. Your spec's avoid residues (metal sites, the Phe88–Phe89–Trp90 face, dimer interface) **cannot** be passed as constraints. The only protection is choosing seeds far from them — which is why the verification cell below checks the computed interface against your avoid set.

## 0. One-time setup — run in a login-node terminal (not in this notebook)

```bash
git clone https://github.com/aqlaboratory/genie3.git
cd genie3
bash scripts/setup/setup.sh          # builds the `genie3` conda env (incl. ColabFold) — heavy, several GB
bash scripts/setup/download.sh       # pretrained weights + data from HuggingFace
conda activate genie3
# then launch Jupyter FROM this env on a login node, and open this notebook
```

In [ ]:
import os

# === EDIT: where you cloned aqlaboratory/genie3 ===
GENIE3_DIR = os.path.expanduser("~/genie3")
os.chdir(GENIE3_DIR)
print("Working dir:", os.getcwd())

# Identifiers / paths (all repo-root-relative, as Genie 3 expects)
PROBLEM_SET  = "sentinel"          # -> name: in the prep config
PROBLEM_KEY  = "s100b"             # -> key under problem:
STAGING_DIR  = f"scripts/problem/binder_design/{PROBLEM_SET}"
TARGET_PDB   = f"{STAGING_DIR}/pdb/s100b_dimer.pdb"
PREP_CONFIG  = f"{STAGING_DIR}/config.yaml"
DATASET_DIR  = f"data/design/binder_design/{PROBLEM_SET}"      # prepare.py writes here
PROBLEM_JSON = f"{DATASET_DIR}/problems/{PROBLEM_KEY}.json"
EXPERIMENT   = f"examples/binder_design/{PROBLEM_SET}/experiment.yaml"

os.makedirs(f"{STAGING_DIR}/pdb", exist_ok=True)
os.makedirs(os.path.dirname(EXPERIMENT), exist_ok=True)

## 1. Environment sanity check (incl. Schrödinger isolation)

Confirm the kernel is in the `genie3` env, has a GPU, and that Schrödinger isn't leaking into the environment (the conflict from earlier — its bundled Python / `LD_LIBRARY_PATH` will break Genie 3's torch).

In [ ]:
import sys, subprocess

print("Python      :", sys.executable)
for v in ["CONDA_DEFAULT_ENV", "SCHRODINGER", "PYTHONPATH", "LD_LIBRARY_PATH"]:
    print(f"{v:16}:", os.environ.get(v, ""))

leaking = [v for v in ["SCHRODINGER", "PYTHONPATH", "LD_LIBRARY_PATH"]
           if "schrodinger" in os.environ.get(v, "").lower()]
print("\nSchrödinger leaking into:", leaking or "none — clean")

try:
    import genie3
    print("genie3 import: OK")
except Exception as e:
    print("genie3 import FAILED — is the kernel in the genie3 env?", e)

print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total",
                      "--format=csv,noheader"], capture_output=True, text=True).stdout or "no GPU on this node")

## 2. Build the target structure

PDB **3D0Y**, chains A+B, HETATM stripped — same target as the BoltzGen run.

> Note: 3D0Y is the Ca²⁺/Zn²⁺-**loaded** conformation. You're avoiding the metal *sites*, but the backbone is still holo. If you want the apo conformation, swap in an apo S100B PDB (e.g. 1B4C) and re-clean.

In [ ]:
import urllib.request

urllib.request.urlretrieve("https://files.rcsb.org/download/3D0Y.pdb", "/tmp/3D0Y.pdb")

clean = []
with open("/tmp/3D0Y.pdb") as f:
    for line in f:
        if line.startswith("ATOM") and line[21] in ("A", "B"):
            clean.append(line)

with open(TARGET_PDB, "w") as f:
    f.writelines(clean)
    f.write("END\n")

print(f"Wrote {len(clean)} ATOM lines -> {TARGET_PDB}")
print("Chains:", sorted(set(l[21] for l in clean)))

## 3. Write the prep config

This is where your spec doc maps. Two things differ from the BoltzGen/Tamarind `hotspot_residues: [...]` style:
- IDs are **chain-prefixed** (`A34`), one per line.
- Hotspots are **seeds**, not the full interface — `prepare.py` expands them by distance. binderbench targets use 3–6 seeds each, so don't list all of 32–39 / 48–59. Pick a few per helix and let the expansion do the rest.

In [ ]:
# Seed hotspots: representative residues on the two outer helices (32-39, 48-59).
HOTSPOTS = ["A34", "A37", "A52", "A55", "A58"]

hotspot_block = "\n".join(f"        {h}" for h in HOTSPOTS)
config_yaml = f"""\
name: {PROBLEM_SET}
problem:
  {PROBLEM_KEY}:
    name: S100B
    target:
      filepath: {TARGET_PDB}
      hotspot:
{hotspot_block}
    binder:
      min_length: 50
      max_length: 80
    tag:
      Sentinel
    other:
      pdb_id: 3D0Y
"""

with open(PREP_CONFIG, "w") as f:
    f.write(config_yaml)
print(config_yaml)

## 4. Prepare the problem set

`prepare.py` runs **ColabFold to build the target MSA** — needs internet (remote MMseqs2 API) or a local ColabFold DB, so run this on a node with network access.

> It refuses to overwrite an existing output dir. To re-prep after editing hotspots, delete it first:
> `rm -rf data/design/binder_design/sentinel`

In [ ]:
!python scripts/problem/binder_design/prepare.py \
    --config {PREP_CONFIG} \
    --outdir data/design/binder_design

## 5. Verify the computed interface against your avoid set ⚠️

The critical check for Project Sentinel: confirm the distance-expanded interface didn't creep into the metal-binding / GFAP-tubulin regions. Your 48–59 seed helix is sequence-adjacent to the 62–73 Ca²⁺ region, so this is a real risk.

In [ ]:
import json

# Avoid set from the spec doc: metal sites 16-19 & 26 & 62-73, high-affinity site 62,
# GFAP/tubulin face 86-91 (Phe88/Phe89/Trp90).
AVOID = set(range(16, 20)) | {26} | set(range(62, 74)) | set(range(86, 92))

prob  = json.load(open(PROBLEM_JSON))
iface = prob["target_interface_residues"]
resnums = lambda tags: sorted(int(t[1:]) for t in tags)   # strip chain letter

print("hotspot :", iface.get("hotspot"))
print("extended:", iface.get("extended"))
print("common  :", iface.get("common"))
print("binder length:", prob.get("binder_min_length"), "-", prob.get("binder_max_length"))

overlap = sorted(set(resnums(iface.get("extended", []))) & AVOID)
if overlap:
    print(f"\n⚠️  extended interface hits AVOID residues: {overlap}")
    print("    -> move hotspots away from these regions, rm -rf the dataset dir, re-run prepare.py")
else:
    print("\n✓  extended interface is clear of all avoid residues")

## 6. Write the run config

In [ ]:
experiment_yaml = f"""\
experiment:
  name: s100b_binders

paths:
  rootdir: examples/binder_design/{PROBLEM_SET}
  dataset: {DATASET_DIR}

generation:
  dataset:
    source: target
    selections: {PROBLEM_KEY}
    n_sample: 100          # per run; for the full 500 use the job array below
  sampler:
    sampler:
      direction_scale: 0.0

evaluation:
  version: binder
  inverse_folding:
    num_seq: 1
  folding:
    model_name: colabfold
    mode: template
"""

with open(EXPERIMENT, "w") as f:
    f.write(experiment_yaml)
print(experiment_yaml)

## 7. Submit the generation to the scheduler

Heavy GPU work goes to Slurm, not the kernel. **Edit the partition/account** to match your cluster. The job script unsets the Schrödinger vars first so they can't break torch.

In [ ]:
sbatch = f"""#!/bin/bash
#SBATCH --job-name=genie3_s100b
#SBATCH --partition=gpu          # <-- EDIT to your GPU partition
#SBATCH --gres=gpu:1
#SBATCH --cpus-per-task=8
#SBATCH --mem=48G
#SBATCH --time=12:00:00
#SBATCH --output=logs/genie3_s100b_%j.out
# #SBATCH --array=0-4            # uncomment for 5x100 = 500 designs (see sharding below)

set -euo pipefail
mkdir -p logs

# --- isolate from Schrödinger so it can't pollute the genie3 env ---
module unload schrodinger 2>/dev/null || true
unset SCHRODINGER PYTHONPATH LD_LIBRARY_PATH

source "$(conda info --base)/etc/profile.d/conda.sh"
conda activate genie3
cd {GENIE3_DIR}

# Single run (100 designs):
genie3 run -c {EXPERIMENT} --num-devices 1

# For the 500-design array instead, comment the line above and use:
# genie3 generate -c {EXPERIMENT} --num-devices 1 \\
#     --shard-id $SLURM_ARRAY_TASK_ID --num-shards 5
# # then ONCE, after all shards finish:
# genie3 evaluate --reduce -c {EXPERIMENT}
"""

with open("run_s100b.sbatch", "w") as f:
    f.write(sbatch)
print("Wrote run_s100b.sbatch — submit with the next cell")

In [ ]:
!sbatch run_s100b.sbatch
# monitor:   !squeue -u $USER
# progress:  !genie3 status -c {EXPERIMENT}

## 8. Load results

After the job finishes. Results land in `<rootdir>/results`; in-silico success is **scRMSD < 2 Å**.

In [ ]:
import pandas as pd

results_dir = f"examples/binder_design/{PROBLEM_SET}/results"
info = pd.read_csv(f"{results_dir}/info.csv")
print(f"{len(info)} designs evaluated")
print("columns:", list(info.columns))

succ = info[info["scrmsd"] < 2.0].sort_values("scrmsd")
print(f"\n{len(succ)} successful (scRMSD < 2 A)")
# successful_generations_cluster.csv has FoldSeek diversity clusters if you want them
succ.head(20)

## 9. Visualize top designs (optional)

Reuses the py3Dmol pattern from the BoltzGen notebook. `colorscheme: chain` handles whatever chain layout Genie 3 writes; recolor per-chain if you want target-vs-binder contrast.

In [ ]:
import py3Dmol, glob

pdbs = sorted(glob.glob(f"{results_dir}/successful_generations/*.pdb"))[:6]
print(f"Showing {len(pdbs)} designs")

ncols = min(len(pdbs), 3) or 1
nrows = (len(pdbs) + ncols - 1) // ncols
view = py3Dmol.view(width=380*ncols, height=320*nrows, viewergrid=(nrows, ncols))
for i, p in enumerate(pdbs):
    r, c = divmod(i, ncols)
    view.addModel(open(p).read(), "pdb", viewer=(r, c))
    view.setBackgroundColor("white", viewer=(r, c))
    view.setStyle({}, {"cartoon": {"colorscheme": "chain"}}, viewer=(r, c))
    view.addLabel(os.path.basename(p).replace(".pdb", ""),
                  {"fontSize": 10, "backgroundOpacity": 0.6}, viewer=(r, c))
    view.zoomTo(viewer=(r, c))
view.show()